# Taller Dirigido
En el presente taller dirigido entrenaremos redes neuronales recurrentes, LSTM y arquitecturas seq-to-seq.

In [1]:
import nltk
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
from sklearn.model_selection import train_test_split

# 1. LSTM y LSTM Bidireccional

En esta primera parte, entrenaremos una red RNN para un problema de PoS Tagging. Para ello, trabajaremos con un corpus Brown.

In [2]:
# Descargar paquetes necesarios para poder rescatar el corpus
nltk.download("brown")
nltk.download("universal_tagset")
from nltk.corpus import brown

#  Extraer oraciones y etiquetas POS del corpus
data = brown.tagged_sents(tagset='universal')
sentences = [[word for word, tag in sent] for sent in data]
pos_tags = [[tag for word, tag in sent] for sent in data]

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


Veamos una muestra de ese conjunto de datos:

In [3]:
data[0]

[('The', 'DET'),
 ('Fulton', 'NOUN'),
 ('County', 'NOUN'),
 ('Grand', 'ADJ'),
 ('Jury', 'NOUN'),
 ('said', 'VERB'),
 ('Friday', 'NOUN'),
 ('an', 'DET'),
 ('investigation', 'NOUN'),
 ('of', 'ADP'),
 ("Atlanta's", 'NOUN'),
 ('recent', 'ADJ'),
 ('primary', 'NOUN'),
 ('election', 'NOUN'),
 ('produced', 'VERB'),
 ('``', '.'),
 ('no', 'DET'),
 ('evidence', 'NOUN'),
 ("''", '.'),
 ('that', 'ADP'),
 ('any', 'DET'),
 ('irregularities', 'NOUN'),
 ('took', 'VERB'),
 ('place', 'NOUN'),
 ('.', '.')]

In [4]:
sentences[0]

['The',
 'Fulton',
 'County',
 'Grand',
 'Jury',
 'said',
 'Friday',
 'an',
 'investigation',
 'of',
 "Atlanta's",
 'recent',
 'primary',
 'election',
 'produced',
 '``',
 'no',
 'evidence',
 "''",
 'that',
 'any',
 'irregularities',
 'took',
 'place',
 '.']

In [5]:
pos_tags[0]

['DET',
 'NOUN',
 'NOUN',
 'ADJ',
 'NOUN',
 'VERB',
 'NOUN',
 'DET',
 'NOUN',
 'ADP',
 'NOUN',
 'ADJ',
 'NOUN',
 'NOUN',
 'VERB',
 '.',
 'DET',
 'NOUN',
 '.',
 'ADP',
 'DET',
 'NOUN',
 'VERB',
 'NOUN',
 '.']

Al igual que en otros talleres, el primer paso consiste en la tokenización de los textos, su secuenciación (de forma numérica) y su padding a una longitud determinada.

In [6]:
word_tokenizer = keras.preprocessing.text.Tokenizer()
tag_tokenizer = keras.preprocessing.text.Tokenizer()

word_tokenizer.fit_on_texts(sentences)
tag_tokenizer.fit_on_texts(pos_tags)

word_index = word_tokenizer.word_index
tag_index = tag_tokenizer.word_index

# Convertir texto a secuencias numéricas
X = word_tokenizer.texts_to_sequences(sentences)
y = tag_tokenizer.texts_to_sequences(pos_tags)
X = [[x - 1 for x in row] for row in X]
y = [[x - 1 for x in row] for row in y]

# Asegurar que todas las secuencias tengan la misma longitud (máximo del dataset)
X = pad_sequences(X, padding="post")
y = pad_sequences(y, padding="post")

# 🔹 Dividir datos en entrenamiento (80%) y validación (20%)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Definimos el modelo neuronal.

In [7]:
model = keras.Sequential([
    keras.layers.Embedding(input_dim=len(word_index) + 1, output_dim=64),
    keras.layers.LSTM(128, return_sequences=True),
    keras.layers.Dense(len(tag_index) + 1, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.fit(X_train, y_train, epochs=1, batch_size = 256, validation_data=(X_val, y_val))

180/180 ━━━━━━━━━━━━━━━━━━━━ 222s 1s/step - accuracy: 0.8994 - loss: 0.3999 - val_accuracy: 0.9717 - val_loss: 0.1014


Probamos el modelo con una nueva oración

In [8]:
new_sentence = "The red fox jumps over the fence"
tokens = new_sentence.split()

new_sequence = word_tokenizer.texts_to_sequences([tokens])[0]
new_sequence = pad_sequences([new_sequence], maxlen=None, padding="post")
new_sequence = np.array([[int(x - 1) for x in row] for row in new_sequence], dtype=np.int32)

In [9]:
predictions = model.predict(new_sequence)

predicted_tags = [list(tag_index.keys())[np.argmax(pred)] for pred in predictions[0]]
for word, tag in zip(tokens, predicted_tags):
    print(f"{word} -> {tag}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
The -> det
red -> noun
fox -> noun
jumps -> noun
over -> adp
the -> det
fence -> noun


Ahora creamos un modelo bidireccional

In [10]:
model2 = keras.Sequential([
    keras.layers.Embedding(input_dim=len(word_index) + 1, output_dim=64),
    keras.layers.Bidirectional(keras.layers.LSTM(128, return_sequences=True)),
    keras.layers.Dense(len(tag_index) + 1, activation="softmax")
])

model2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model2.fit(X_train, y_train, epochs=1, batch_size = 128, validation_data=(X_val, y_val))

359/359 ━━━━━━━━━━━━━━━━━━━━ 471s 1s/step - accuracy: 0.9242 - loss: 0.2823 - val_accuracy: 0.9919 - val_loss: 0.0283


In [11]:
predictions = model2.predict(new_sequence)

predicted_tags = [list(tag_index.keys())[np.argmax(pred)] for pred in predictions[0]]
for word, tag in zip(tokens, predicted_tags):
    print(f"{word} -> {tag}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 350ms/step
The -> det
red -> adj
fox -> noun
jumps -> noun
over -> adp
the -> det
fence -> noun


# 2. Seq-to-Seq

Creamos un corpus paralelo de textos en inglés y en francés

In [12]:
english_sentences = [
    "Hello, how are you?", "The weather is nice today.", "She loves reading books.",
    "I need a cup of coffee.", "We are learning artificial intelligence.", "Dogs are very loyal animals.",
    "Where is the nearest restaurant?", "The train leaves at 5 PM.", "He plays the piano beautifully.",
    "Can you help me with this task?", "I watched a great movie yesterday.", "This place is amazing!",
    "Technology evolves rapidly.", "The Eiffel Tower is in Paris.", "My favorite fruit is mango.",
    "Music makes me feel relaxed.", "She is studying neuroscience.", "I need to buy some groceries.",
    "There is a bakery near my house.", "The sun is shining brightly.", "I love playing the guitar.",
    "My computer is very fast.", "We traveled to Spain last year.", "The book was interesting.",
    "He enjoys hiking in the mountains.", "Can you recommend a good restaurant?", "She speaks three languages.",
    "The car is parked outside.", "I prefer tea over coffee.", "The concert was fantastic!",
    "This hotel has a great view.", "I am learning a new programming language.", "The museum is very famous.",
    "He is writing a novel.", "She is painting a beautiful picture.", "We are planning a trip to Italy.",
    "The child is playing with toys.", "I will cook dinner tonight.", "The train arrived on time.",
    "The doctor gave me good advice.", "The sky is full of stars.", "My favorite animal is the elephant.",
    "She is swimming in the pool.", "The flowers are blooming.", "They are watching a football match.",
    "I lost my keys yesterday.", "The airplane is ready for departure.", "We are going to the cinema.",
    "The school starts at 8 AM.", "My phone battery is low.", "She is looking for a new apartment.",
    "He found an interesting article online.", "The cake is delicious.", "I saw a rainbow after the rain.",
    "We are listening to music.", "He is repairing his bicycle.", "She is a great dancer.",
    "The city is crowded.", "This dog is very playful.", "They are discussing an important topic.",
    "I will call you tomorrow.", "She gave me a wonderful gift.", "The temperature is very high.",
    "We need to find a solution.", "The house is made of wood.", "She is drinking a cup of tea.",
    "The movie starts at 7 PM.", "I read a book about history.", "The festival was amazing.",
    "My backpack is heavy.", "He visited his grandparents last weekend.", "She is writing a letter.",
    "This place is magical.", "The beach is beautiful in summer.", "He is drawing a landscape.",
    "They are learning a new skill.", "We arrived at the airport early.", "The bird is flying over the lake.",
    "She baked a chocolate cake.", "The train station is near my house.", "He bought a new pair of shoes.",
    "The water is cold.", "She is wearing a red dress.", "The kids are playing in the park.",
    "My brother loves science.", "The restaurant serves delicious food.", "They built a new bridge.",
    "This car is very expensive.", "He found a good job.", "We are visiting a historic site.",
    "The flight was comfortable.", "She is singing a famous song.", "The stars are shining brightly."
]
french_sentences = [
    "Bonjour, comment ça va ?", "Le temps est agréable aujourd'hui.", "Elle adore lire des livres.",
    "J'ai besoin d'une tasse de café.", "Nous apprenons l'intelligence artificielle.", "Les chiens sont des animaux très fidèles.",
    "Où est le restaurant le plus proche ?", "Le train part à 17h.", "Il joue du piano magnifiquement.",
    "Peux-tu m'aider avec cette tâche ?", "J'ai regardé un excellent film hier.", "Cet endroit est incroyable !",
    "La technologie évolue rapidement.", "La tour Eiffel est à Paris.", "Mon fruit préféré est la mangue.",
    "La musique me rend détendu.", "Elle étudie les neurosciences.", "J'ai besoin d'acheter des provisions.",
    "Il y a une boulangerie près de chez moi.", "Le soleil brille intensément.", "J'aime jouer de la guitare.",
    "Mon ordinateur est très rapide.", "Nous avons voyagé en Espagne l'année dernière.", "Le livre était intéressant.",
    "Il aime faire de la randonnée en montagne.", "Peux-tu recommander un bon restaurant ?", "Elle parle trois langues.",
    "La voiture est garée dehors.", "Je préfère le thé au café.", "Le concert était fantastique !",
    "Cet hôtel offre une belle vue.", "J'apprends un nouveau langage de programmation.", "Le musée est très célèbre.",
    "Il écrit un roman.", "Elle peint un tableau magnifique.", "Nous prévoyons un voyage en Italie.",
    "L'enfant joue avec des jouets.", "Je vais cuisiner ce soir.", "Le train est arrivé à l'heure.",
    "Le docteur m'a donné de bons conseils.", "Le ciel est rempli d'étoiles.", "Mon animal préféré est l'éléphant.",
    "Elle nage dans la piscine.", "Les fleurs sont en pleine floraison.", "Ils regardent un match de football.",
    "J'ai perdu mes clés hier.", "L'avion est prêt à décoller.", "Nous allons au cinéma.",
    "L'école commence à 8h.", "La batterie de mon téléphone est faible.", "Elle cherche un nouvel appartement.",
    "Il a trouvé un article intéressant en ligne.", "Le gâteau est délicieux.", "J'ai vu un arc-en-ciel après la pluie.",
    "Nous écoutons de la musique.", "Il répare son vélo.", "Elle est une excellente danseuse.",
    "La ville est bondée.", "Ce chien est très joueur.", "Ils discutent d'un sujet important.",
    "Je t'appellerai demain.", "Elle m'a offert un cadeau merveilleux.", "La température est très élevée.",
    "Nous devons trouver une solution.", "La maison est en bois.", "Elle boit une tasse de thé.",
    "Le film commence à 19h.", "J'ai lu un livre sur l'histoire.", "Le festival était incroyable.",
    "Mon sac à dos est lourd.", "Il a rendu visite à ses grands-parents le week-end dernier.", "Elle écrit une lettre.",
    "Cet endroit est magique.", "La plage est belle en été.", "Il dessine un paysage.",
    "Ils apprennent une nouvelle compétence.", "Nous sommes arrivés tôt à l'aéroport.", "L'oiseau vole au-dessus du lac.",
    "Elle a fait un gâteau au chocolat.", "La gare est près de chez moi.", "Il a acheté une nouvelle paire de chaussures.",
    "L'eau est froide.", "Elle porte une robe rouge.", "Les enfants jouent dans le parc.",
    "Mon frère aime la science.", "Le restaurant sert de la nourriture délicieuse.", "Ils ont construit un nouveau pont.",
    "Cette voiture est très chère.", "Il a trouvé un bon emploi.", "Nous visitons un site historique.",
    "Le vol était confortable.", "Elle chante une chanson célèbre.", "Les étoiles brillent intensément."
]

Para el caso del idioma destino, añadimos un marcador de inicio y fin de oración.

In [13]:
french_sentences = [f"<START> {sentence} <END>" for sentence in french_sentences]

In [14]:
french_sentences[0]

'<START> Bonjour, comment ça va ? <END>'

Se tokeniza el texto y se genera la secuencia, como hemos trabajado anteriormente:

In [15]:
# Tokenización
from tensorflow.keras.preprocessing.text import Tokenizer
eng_tokenizer = Tokenizer()
fr_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)
fr_tokenizer.fit_on_texts(french_sentences)
eng_vocab_size = len(eng_tokenizer.word_index) + 1
fr_vocab_size = len(fr_tokenizer.word_index) + 1

# Secueniamiento
X = eng_tokenizer.texts_to_sequences(english_sentences)
y = fr_tokenizer.texts_to_sequences(french_sentences)
X = pad_sequences(X, maxlen=None, padding="post")
y = pad_sequences(y, maxlen=None, padding="post")

# Distribución
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Se define la arquitectura del encoder. Para ellos usamos el paradigma funcional y no el secuencial.

In [16]:
# ENCODER
encoder_inputs = keras.layers.Input(shape=(X.shape[1],))
encoder_embedding = keras.layers.Embedding(eng_vocab_size, 64)(encoder_inputs)
encoder_lstm = keras.layers.Bidirectional(keras.layers.LSTM(128, return_sequences=False))(encoder_embedding)
encoder_output = keras.layers.Dense(128, activation="relu")(encoder_lstm)

# DECODER
decoder_inputs = keras.layers.Input(shape=(y.shape[1],))
decoder_embedding = keras.layers.Embedding(fr_vocab_size, 64)(decoder_inputs)
# El argumento initial_state recibe el estado oculto inicial (que viene del encoder) y también la información contextual inicial (que viene del encoder)
decoder_lstm = keras.layers.LSTM(128, return_sequences=True)(decoder_embedding, initial_state=[encoder_output, encoder_output])
decoder_output = keras.layers.Dense(fr_vocab_size, activation="softmax")(decoder_lstm)

# MODELO GLOBAL
# Define que el modelo tiene 2 capas de entrada: lo que entra al encoder y al decoder (lista)
# Y también que tiene 1 capa de salida: la salida del decoder.
model3 = keras.Model([encoder_inputs, decoder_inputs], decoder_output)

In [17]:
model3.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model3.fit([X_train, y_train], y_train, epochs=10, validation_data=([X_val, y_val], y_val))

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 350ms/step - accuracy: 0.2109 - loss: 5.6362 - val_accuracy: 0.5376 - val_loss: 5.5749
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5426 - loss: 5.5553 - val_accuracy: 0.4962 - val_loss: 5.4405
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.4813 - loss: 5.3968 - val_accuracy: 0.4662 - val_loss: 5.1361
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.4785 - loss: 5.0188 - val_accuracy: 0.4662 - val_loss: 4.3824
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.4645 - loss: 4.1609 - val_accuracy: 0.4662 - val_loss: 3.2556
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.4704 - loss: 3.1248 - val_accuracy: 0.4662 - val_loss: 3.1039
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.4628 - loss: 3.1268 - val_accuracy: 0.4662 - val_loss: 3.2172
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.4676 - loss: 3.1294 - val_accuracy: 0.4662 - val_loss: 3.0709

Probamos el modelo aprendido con un texto

In [18]:
# Tokenización y secuenciamiento
test_sentence = "The sky is blue"
tokens = test_sentence.split()
test_sequence = eng_tokenizer.texts_to_sequences([tokens])[0]
# Es importante indicar que la longitud máxima es la encontrada en el set de entrenamiento.
test_sequence = pad_sequences([test_sequence], maxlen=X_train.shape[1], padding="post")

# Definimos el valor de entrada para el decoder, que es el token de inicio de oración
start_token = fr_tokenizer.word_index.get("<START> ", 1)
# Se define la entrada del decoder como una matriz de 1 fila y tantas columnas como longitud máxima tiene el lenguaje destino
decoder_input = np.zeros((1, y_train.shape[1]))
# Inicia la secuencia con el token de inicio (pone su posición en 1)
decoder_input[0, 0] = start_token

translated_words = []
for i in range(y_train.shape[1] - 1):
    # Se alimenta la red con el texto en ingles, y el estado inicial del decoder
    # Predictions tiene 3 dimensiones: batch_size x sequence_length x vocab_size
    predictions = model3.predict([test_sequence, decoder_input])
    next_word_idx = np.argmax(predictions[0, i])
    translated_words.append(list(fr_tokenizer.word_index.keys())[next_word_idx])

    # Agregar la palabra generada al input del decoder
    decoder_input[0, i + 1] = next_word_idx
    if list(fr_tokenizer.word_index.keys())[next_word_idx] == 'end':
      break

# 🔹 Mostrar resultados
translated_sentence = " ".join(translated_words)
print(f"Traducción generada: {translated_sentence}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Traducción generada: est est end
